In [ ]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from gwpy.timeseries import TimeSeries
from ripplegw.waveforms.cbc.IMRPhenomD import IMRPhenomD

sys.path.append("../src")
from filtro_adaptado import psd_en, serie_snr, snr_optimo, transformada
from senal import irfft

jax.config.update("jax_enable_x64", True)

In [ ]:
EVENTO = "GW150914"
DATOS = Path("../datos")
DURACION = 32
DURACION_FFT = 4
MEDIO_SEGMENTO = 4
F_MIN = 20
F_MAX = 1024
VENTANA = 0.5
MASAS = np.round(np.arange(24, 40.01, 0.2), 2)
ETAS = [0.25, 0.24, 0.2222]
GIROS = np.round(np.arange(-0.35, 0.2501, 0.05), 2)
MASA_CHIRP_CATALOGO = 30.2
POSICIONES = [-11, -9, -7, -5, -3, 3, 5, 7, 9, 11]
MASA_INYECTADA = 30
ETA_INYECTADA = 0.2469
SNR_INYECTADO = {"H1": 22.3, "L1": 15.1}

In [ ]:
t0 = float((DATOS / f"{EVENTO}_gps.txt").read_text())
datos_h1 = TimeSeries.read(DATOS / f"H1_{EVENTO}_{DURACION}s.hdf5")
datos_l1 = TimeSeries.read(DATOS / f"L1_{EVENTO}_{DURACION}s.hdf5")
fs = int(datos_h1.sample_rate.value)
tiempos = datos_h1.times.value - t0
crudos = {"H1": datos_h1.value, "L1": datos_l1.value}
n = 2 * MEDIO_SEGMENTO * fs
df = fs / n
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)

In [ ]:
def generador(frecuencias):
    f = jnp.array(frecuencias)
    funcion = jax.jit(lambda parametros: IMRPhenomD.gen_IMRPhenomD(f, parametros, F_MIN))
    return lambda masa_chirp, eta, giro: np.asarray(funcion(jnp.array([masa_chirp, eta, giro, giro, 1.0, 0.0, 0.0])))


plantilla = generador(indices * df)

In [ ]:
def snr_maximo(crudo, centro, parametros):
    psd = psd_en(np.arange(n // 2 + 1) * df, crudo, fs, DURACION_FFT)
    inicio = np.searchsorted(tiempos, centro - MEDIO_SEGMENTO)
    datos_f = transformada(crudo[inicio : inicio + n], fs, fs // 2)
    t_segmento = tiempos[inicio : inicio + n]
    cerca = np.abs(t_segmento - centro) < VENTANA
    snr = np.zeros(len(parametros))
    instante = np.zeros(len(parametros))
    for a in range(0, len(parametros), 100):
        plantillas = np.array([plantilla(*p) for p in parametros[a : a + 100]])
        serie = serie_snr(datos_f, plantillas, psd, df, n, indices)[:, cerca]
        snr[a : a + 100] = serie.max(axis=1)
        instante[a : a + 100] = t_segmento[cerca][serie.argmax(axis=1)]
    return snr, instante


def distribucion(snr2, valores):
    unicos = np.unique(valores)
    peso = np.exp(snr2 / 2 - snr2.max() / 2)
    probabilidad = np.array([peso[valores == u].sum() for u in unicos])
    return unicos, probabilidad / probabilidad.sum()


def intervalo(unicos, probabilidad):
    acumulada = np.cumsum(probabilidad)
    return [unicos[np.searchsorted(acumulada, q)] for q in (0.05, 0.5, 0.95)]

In [ ]:
parametros = [(m, e, g) for g in GIROS for e in ETAS for m in MASAS]
masas = np.array([p[0] for p in parametros])
giros = np.array([p[2] for p in parametros])
snr_h1, instante_h1 = snr_maximo(crudos["H1"], 0, parametros)
snr_l1, instante_l1 = snr_maximo(crudos["L1"], 0, parametros)

In [ ]:
for nombre, snr2 in [("H1", snr_h1**2), ("L1", snr_l1**2), ("H1+L1", snr_h1**2 + snr_l1**2)]:
    bajo, mediana, alto = intervalo(*distribucion(snr2, masas))
    g_bajo, g_mediana, g_alto = intervalo(*distribucion(snr2, giros))
    print(f"{nombre}: SNR {np.sqrt(snr2.max()):.1f}, masa chirp {mediana:.1f} M☉ (90 %: {bajo:.1f} a {alto:.1f}), {100 * (mediana / MASA_CHIRP_CATALOGO - 1):+.1f} % del catálogo, giro {g_mediana:+.2f} ({g_bajo:+.2f} a {g_alto:+.2f})")
mejor = np.argmax(snr_h1**2 + snr_l1**2)
print(f"La señal llega a L1 {1000 * (instante_h1[mejor] - instante_l1[mejor]):.1f} ms antes que a H1")

In [ ]:
for nombre, snr2 in [("H1", snr_h1**2), ("L1", snr_l1**2), ("H1+L1", snr_h1**2 + snr_l1**2)]:
    plt.plot(*distribucion(snr2, masas), label=nombre)
plt.axvline(MASA_CHIRP_CATALOGO, color="k", linestyle="--", label="catálogo")
plt.xlabel(r"Masa chirp (M$_\odot$)")
plt.ylabel("Probabilidad")
plt.legend();

In [ ]:
mapa = (snr_h1**2 + snr_l1**2).reshape(len(GIROS), len(ETAS), len(MASAS)).max(axis=1)
plt.pcolormesh(MASAS, GIROS, np.sqrt(mapa), shading="auto")
plt.colorbar(label="SNR de red")
plt.xlabel(r"Masa chirp (M$_\odot$)")
plt.ylabel("Giro");

In [ ]:
n_total = DURACION * fs
df_total = fs / n_total
indices_total = np.arange(int(15 / df_total), int(F_MAX / df_total) + 1)
forma_inyectada = generador(indices_total * df_total)(MASA_INYECTADA, ETA_INYECTADA, 0.0)
en_banda = indices_total * df_total >= F_MIN


def inyectar(ifo, posicion):
    psd = psd_en(np.arange(n_total // 2 + 1) * df_total, crudos[ifo], fs, DURACION_FFT)
    escala = SNR_INYECTADO[ifo] / snr_optimo(forma_inyectada[en_banda], psd, df_total, indices_total[en_banda])
    espectro = np.zeros(n_total // 2 + 1, dtype=complex)
    espectro[indices_total] = escala * forma_inyectada * np.exp(-2j * np.pi * indices_total * df_total * (posicion - tiempos[0]))
    return crudos[ifo] + irfft(espectro * fs)

In [ ]:
parametros_sin_giro = [(m, e, 0.0) for e in ETAS for m in MASAS]
masas_sin_giro = np.array([p[0] for p in parametros_sin_giro])
errores = []
cubre = 0
for posicion in POSICIONES:
    snr2 = sum(snr_maximo(inyectar(ifo, posicion), posicion, parametros_sin_giro)[0] ** 2 for ifo in ["H1", "L1"])
    bajo, mediana, alto = intervalo(*distribucion(snr2, masas_sin_giro))
    errores.append(100 * (mediana / MASA_INYECTADA - 1))
    cubre += bajo <= MASA_INYECTADA <= alto
errores = np.array(errores)
print(f"Sesgo {errores.mean():+.1f} %, dispersión {errores.std(ddof=1):.1f} %, peor {errores[np.argmax(np.abs(errores))]:+.1f} %, dentro del 5 %: {(np.abs(errores) < 5).sum()}/{len(errores)}, el 90 % contiene la masa verdadera: {cubre}/{len(errores)}")

In [ ]:
plt.plot(POSICIONES, errores, "o")
plt.axhspan(-5, 5, color="0.9")
plt.axhline(0, color="k")
plt.xlabel("Posición de la inyección (s desde t0)")
plt.ylabel("Error de la masa chirp (%)");